Code that runs QAOA (arXiv:1411.4028), DC-QAOA (arXiv:2107.02789), CD-inspired (arXiv:2212.13511), CD-mixer (arXiv:2502.15375), and CD-prob for the Schwinger model. It runs the algorithms for a certain number of runs and a range of layers, with different random parameters each time. It then plots the average energy plots, the best energy plots (the run that reached the lowest energy), average approximation ratios, best approximation ratios (the ones closest to one), average state fidelity, and the best state fidelity (the ones closest to one). The average plots for the DC-QAOA, CD-inspired, CD-mixer, and CD-prob only include the runs with the fixed A_DC term.

Saves the plots of the data and text files containing data from the running the algorithms. If you do not want the plots or text files saved comment out those sections.

With the current variables set as they are, the code will take about a day to run.

In [ ]:

# Imports the modules.

import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numpy as np
from scipy.optimize import minimize
from datetime import datetime
import sys
 
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.parametervector import ParameterVector
from qiskit.circuit.quantumcircuit import QuantumCircuit
import qiskit_algorithms.optimizers as opt
from qiskit.quantum_info import Statevector, state_fidelity

from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Aer_Estimator

In [ ]:

# Function that runs QAOA.
# Inputs problem Hamiltonian (H_prob), mixer Hamiltonian (H_mix), mixer ground state (initial_state), number of layers in circuit (layer), 
# initial variational parameter values (init_params), number of qubits (num_qubits), and name of optimizer to use (optimizer).
# Optimizers that you can use are GradientDescent, COBYLA, and SPSA from the qiskit algorithms library.
# Need to define global variable opt_vals_QAOA, parameter_vals_QAOA and minimum_value_QAOA before running which stores the cost and parameter values.
# Returns result found by optimizer and a statevector of the final ansatz state.

def QAOA(H_prob, H_mix, initial_state, layer, init_params, num_qubits, optimizer):
    
    # Defines the sets of variational parameters.
    gammas = ParameterVector('γ', layer)
    betas  = ParameterVector('β', layer)
    
    # Builds the ansatz state circuit.
    QAOA_Ansatz = initial_state.copy()
    for i in range(layer):
        # Applys problem Hamiltonian unitary.
        gate_prob = PauliEvolutionGate(H_prob, time=gammas[i])
        QAOA_Ansatz.append(gate_prob, range(num_qubits))
        # Applys mixer Hamiltonian unitary.
        gate_mix = PauliEvolutionGate(H_mix, time=betas[i])
        QAOA_Ansatz.append(gate_mix, range(num_qubits))
    
    # Uses AerSimulator as the backed.
    backend = AerSimulator()

    # Seeding the algorithm for better testing.
    seed = 100 
    # Creates pass manager for transpilation.
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=seed) 
    candidate_circuit = pm.run(QAOA_Ansatz)

    # Expectation values are calculated using Aer_Estimator. It calculates expectation values exactly.
    estimator = Aer_Estimator()

    # Defines expectation value F_P.
    def cost_func_estimator(params):
        pub = (candidate_circuit, H_prob, params)
        job = estimator.run([pub])
        results = job.result()[0]
        cost = float(results.data.evs)
        return cost

    # Callback functions for the optimizers.
    # Saves expectation value F_P values during the optimization to opt_values_QAOA, and parameter values to parameter_values_QAOA.

    # Callback function for GradientDescent. 
    def callback_fun_GradientDescent(eval_count, parameters, value, norm):    
        opt_values_QAOA.append(value)
        parameter_values_QAOA.append(str(parameters).replace("\n", ""))
    
    # Callback function for COBYLA.
    def callback_fun_COBYLA(xk):
        value = cost_func_estimator(xk)
        opt_values_QAOA.append(value)
        parameter_values_QAOA.append(str(parameters).replace("\n", ""))

    # Callback function for SPSA.
    def callback_fun_SPSA(eval_count, parameters, value, stepsize, step_accepted):    
        opt_values_QAOA.append(value)
        parameter_values_QAOA.append(str(parameters).replace("\n", ""))
    
    # Gets the callback function for the respective optimizer.
    optimizer.callback = locals().get(f"callback_fun_{optimizer.__class__.__name__}")
    # Sets the optimizer to minimize the expectation value F_P for a given initial point x0.
    result = optimizer.minimize(fun = cost_func_estimator, x0 = init_params)
    # Converts final ansatz state circuit into a statevector.
    QAOA_state = Statevector.from_instruction(candidate_circuit.assign_parameters(result.x))

    return result, QAOA_state

In [ ]:

# Function that runs DC-QAOA.
# Inputs problem Hamiltonian (H_prob), mixer Hamiltonian (H_mix), set of A_DC operators (A_DC), mixer ground state (initial_state), 
# number of layers in circuit (layer), initial variational parameter values (init_params), number of qubits (num_qubits), 
# and name of optimizer to use (optimizer).
# Optimizers that you can use are GradientDescent, COBYLA, and SPSA from the qiskit algorithms library.
# Need to define global variable opt_vals_DC_QAOA, parameter_vals_DC_QAOA and minimum_value_DC_QAOA before running which stores the cost 
# and parameter values.
# Returns result found by optimizer and a statevector of the final ansatz state.

def DC_QAOA(H_prob, H_mix, A_DC, initial_state, layer, init_params, num_qubits, optimizer):
    
    # Defines the sets of variational parameters.
    gammas = ParameterVector('γ', layer)
    betas  = ParameterVector('β', layer)
    alphas = ParameterVector('α', layer)
    
    # Builds the ansatz state circuit.
    DC_QAOA_Ansatz = initial_state.copy()
    for i in range(layer):
        # Applys problem Hamiltonian unitary.
        gate_prob = PauliEvolutionGate(H_prob, time=gammas[i])
        DC_QAOA_Ansatz.append(gate_prob, range(num_qubits))
        # Applys mixer Hamiltonian unitary.
        gate_mix = PauliEvolutionGate(H_mix, time=betas[i])
        DC_QAOA_Ansatz.append(gate_mix, range(num_qubits))
        # Applys A_DC unitary.
        gate_CD = PauliEvolutionGate(A_DC[i], time=alphas[i])
        DC_QAOA_Ansatz.append(gate_CD, range(num_qubits))

    # Uses AerSimulator as the backed.
    backend = AerSimulator()

    # Seeding the algorithm for better testing.
    seed = 100 
    # Creates pass manager for transpilation.
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=seed) 
    candidate_circuit = pm.run(DC_QAOA_Ansatz)

    # Expectation values are calculated using Aer_Estimator. It calculates expectation values exactly.
    estimator = Aer_Estimator()

    # Defines expectation value F_P.
    def F_P_estimator(params):
        pub = (candidate_circuit, H_prob, params)
        job = estimator.run([pub])
        results = job.result()[0]
        cost = float(results.data.evs)
        return cost

    # Callback functions for the optimizers.
    # Saves F_P values during the optimization to opt_values_DC_QAOA, and parameter values to parameter_values_DC_QAOA.

    # Callback function for GradientDescent. 
    def callback_fun_GradientDescent(eval_count, parameters, value, norm):    
        opt_values_DC_QAOA.append(value)
        parameter_values_DC_QAOA.append(str(parameters).replace("\n", ""))
    
    # Callback function for COBYLA
    def callback_fun_COBYLA(xk):
        value = F_P_estimator(xk)
        opt_values_DC_QAOA.append(value)
        parameter_values_DC_QAOA.append(str(parameters).replace("\n", ""))

    # Callback function for SPSA
    def callback_fun_SPSA(eval_count, parameters, value, stepsize, step_accepted):    
        opt_values_DC_QAOA.append(value)
        parameter_values_DC_QAOA.append(str(parameters).replace("\n", ""))
    
    # Gets the callback function for the respective optimizer.
    optimizer.callback = locals().get(f"callback_fun_{optimizer.__class__.__name__}")
    # Sets the optimizer to minimize the expectation value F_P for a given initial point x0.
    result = optimizer.minimize(fun = F_P_estimator, x0 = init_params)
    # Converts final ansatz state circuit into a statevector.
    DC_QAOA_state = Statevector.from_instruction(candidate_circuit.assign_parameters(result.x))

    return result, DC_QAOA_state

In [ ]:

# Function that runs CD-inspired.
# Inputs problem Hamiltonian (H_prob), set of A_DC operators (A_DC), mixer ground state (initial_state), number of layers in circuit (layer), 
# initial variational parameter values (init_params), number of qubits (num_qubits), and name of optimizer to use (optimizer).
# Optimizers that you can use are GradientDescent, COBYLA, and SPSA from the qiskit algorithms library.
# Need to define global variable opt_vals_CD_inspired, parameter_vals_CD_inspired and minimum_value_CD_inspired before running which stores the cost 
# and parameter values.
# Returns result found by optimizer and a statevector of the final ansatz state.

def CD_inspired(H_prob, A_DC, initial_state, layer, init_params, num_qubits, optimizer):
    
    # Defines the sets of variational parameters.
    alphas = ParameterVector('α', layer)
    
    # Builds the ansatz state circuit.
    CD_inspired_Ansatz = initial_state.copy()
    for i in range(layer):
        # Applys the A_DC unitary.
        gate_CD = PauliEvolutionGate(A_DC[i], time=alphas[i])
        CD_inspired_Ansatz.append(gate_CD, range(num_qubits))

    # Uses AerSimulator as the backed.
    backend = AerSimulator()

    # Seeding the algorithm for better testing.
    seed = 100 
    # Creates pass manager for transpilation.
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=seed) 
    candidate_circuit = pm.run(CD_inspired_Ansatz)

    # Expectation values are calculated using Aer_Estimator. It calculates expectation values exactly.
    estimator = Aer_Estimator()

    # Defines expectation value F_P.
    def F_P_estimator(params):
        pub = (candidate_circuit, H_prob, params)
        job = estimator.run([pub])
        results = job.result()[0]
        cost = float(results.data.evs)
        return cost

    # Callback functions for the optimizers.
    # Saves F_P values during the optimization to opt_values_CD_inspired, and parameter values to parameter_values_CD_inspired.

    # Callback function for GradientDescent. 
    def callback_fun_GradientDescent(eval_count, parameters, value, norm):    
        opt_values_CD_inspired.append(value)
        parameter_values_CD_inspired.append(str(parameters).replace("\n", ""))
    
    # Callback function for COBYLA.
    def callback_fun_COBYLA(xk):
        value = F_P_estimator(xk)
        opt_values_CD_inspired.append(value)
        parameter_values_CD_inspired.append(str(parameters).replace("\n", ""))

    # Callback function for SPSA.
    def callback_fun_SPSA(eval_count, parameters, value, stepsize, step_accepted):    
        opt_values_CD_inspired.append(value)
        parameter_values_CD_inspired.append(str(parameters).replace("\n", ""))
    
    # Gets the callback function for the respective optimizer.
    optimizer.callback = locals().get(f"callback_fun_{optimizer.__class__.__name__}")
    # Sets the optimizer to minimize the expectation value F_P for a given initial point x0.
    result = optimizer.minimize(fun = F_P_estimator, x0 = init_params)
    # Converts final ansatz state circuit into a statevector.
    CD_inspired_state = Statevector.from_instruction(candidate_circuit.assign_parameters(result.x))

    return result, CD_inspired_state

In [ ]:

# Function that runs CD-mixer.
# Inputs problem Hamiltonian (H_prob), mixer Hamiltonian (H_mix) set of A_DC operators (A_DC), mixer ground state (initial_state), 
# number of layers in circuit (layer), initial variational parameter values (init_params), number of qubits (num_qubits), 
# and name of optimizer to use (optimizer).
# Optimizers that you can use are GradientDescent, COBYLA, and SPSA from the qiskit algorithms library.
# Need to define global variable opt_vals_CD_mixer, parameter_vals_CD_mixer and minimum_value_CD_mixer before running which stores the cost 
# and parameter values.
# Returns result found by optimizer and a statevector of the final ansatz state.

def CD_mixer(H_prob, H_mix, A_DC, initial_state, layer, init_params, num_qubits, optimizer):
    
    # Defines the sets of variational parameters.
    betas  = ParameterVector('β', layer)
    alphas = ParameterVector('α', layer)
    
    # Building the circuit.
    CD_mixer_Ansatz = initial_state.copy()
    for i in range(layer):
        # Applys mixer Hamiltonian unitary.
        gate_mix = PauliEvolutionGate(H_mix, time=betas[i])
        CD_mixer_Ansatz.append(gate_mix, range(num_qubits))
        # Applys the A_DC unitary.
        gate_CD = PauliEvolutionGate(A_DC[i], time=alphas[i])
        CD_mixer_Ansatz.append(gate_CD, range(num_qubits))

    # Uses AerSimulator as the backed.
    backend = AerSimulator()

    # Seeding the algorithm for better testing.
    seed = 100 
    # Creates pass manager for transpilation.
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=seed) 
    candidate_circuit = pm.run(CD_mixer_Ansatz)

    # Expectation values are calculated using Aer_Estimator. It calculates expectation values exactly.
    estimator = Aer_Estimator()

    # Defines expectation value F_P.
    def F_P_estimator(params):
        pub = (candidate_circuit, H_prob, params)
        job = estimator.run([pub])
        results = job.result()[0]
        cost = float(results.data.evs)
        return cost

    # Callback functions for the optimizers.
    # Saves F_P values during the optimization to opt_values_CD_mixer, and parameter values to parameter_values_CD_mixer.

    # Callback function for GradientDescent. 
    def callback_fun_GradientDescent(eval_count, parameters, value, norm):    
        opt_values_CD_mixer.append(value)
        parameter_values_CD_mixer.append(str(parameters).replace("\n", ""))
    
    # Callback function for COBYLA.
    def callback_fun_COBYLA(xk):
        value = F_P_estimator(xk)
        opt_values_CD_mixer.append(value)
        parameter_values_CD_mixer.append(str(parameters).replace("\n", ""))

    # Callback function for SPSA.
    def callback_fun_SPSA(eval_count, parameters, value, stepsize, step_accepted):    
        opt_values_CD_mixer.append(value)
        parameter_values_CD_mixer.append(str(parameters).replace("\n", ""))
    
    # Gets the callback function for the respective optimizer.
    optimizer.callback = locals().get(f"callback_fun_{optimizer.__class__.__name__}")
    # Sets the optimizer to minimize the expectation value F_P for a given initial point x0.
    result = optimizer.minimize(fun = F_P_estimator, x0 = init_params)
    # Converts final ansatz state circuit into a statevector.
    CD_mixer_state = Statevector.from_instruction(candidate_circuit.assign_parameters(result.x))

    return result, CD_mixer_state

In [ ]:

# Function that runs CD-prob.
# Inputs problem Hamiltonian (H_prob), set of A_DC operators (A_DC), mixer ground state (initial_state), number of layers in circuit (layer), 
# initial variational parameter values (init_params), number of qubits (num_qubits), and name of optimizer to use (optimizer).
# Optimizers that you can use are GradientDescent, COBYLA, and SPSA from the qiskit algorithms library.
# Need to define global variable opt_vals_QAOA, parameter_vals_CD_prob and minimum_value_CD_prob before running which stores the cost 
# and parameter values.
# Returns result found by optimizer and a statevector of the final ansatz state.

def CD_prob(H_prob, A_DC, initial_state, layer, init_params, num_qubits, optimizer):
    
    # Defines the sets of variational parameters.
    gammas = ParameterVector('γ', layer)
    betas  = ParameterVector('β', layer)
    alphas = ParameterVector('α', layer)
    
    # Building the circuit
    CD_prob_Ansatz = initial_state.copy()
    for i in range(layer):
        # Applys problem Hamiltonian unitary.
        gate_prob = PauliEvolutionGate(H_prob, time=gammas[i])
        CD_prob_Ansatz.append(gate_prob, range(num_qubits))
        # Applys the A_DC unitary.
        gate_CD = PauliEvolutionGate(A_DC[i], time=alphas[i])
        CD_prob_Ansatz.append(gate_CD, range(num_qubits))

    # Uses AerSimulator as the backed.
    backend = AerSimulator()

    # Seeding the algorithm for better testing.
    seed = 100 
    # Creates pass manager for transpilation.
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=seed) 
    candidate_circuit = pm.run(CD_prob_Ansatz)

    # Expectation values are calculated using Aer_Estimator. It calculates expectation values exactly.
    estimator = Aer_Estimator()

    # Defines expectation value F_P.
    def F_P_estimator(params):
        pub = (candidate_circuit, H_prob, params)
        job = estimator.run([pub])
        results = job.result()[0]
        cost = float(results.data.evs)
        return cost

    # Callback functions for the optimizers.
    # Saves F_P values during the optimization to opt_values_CD_prob, and parameter values to parameter_values_CD_prob.

    # Callback function for GradientDescent. 
    def callback_fun_GradientDescent(eval_count, parameters, value, norm):    
        opt_values_CD_prob.append(value)
        parameter_values_CD_prob.append(str(parameters).replace("\n", ""))
    
    # Callback function for COBYLA.
    def callback_fun_COBYLA(xk):
        value = F_P_estimator(xk)
        opt_values_CD_prob.append(value)
        parameter_values_CD_prob.append(str(parameters).replace("\n", ""))

    # Callback function for SPSA.
    def callback_fun_SPSA(eval_count, parameters, value, stepsize, step_accepted):    
        opt_values_CD_prob.append(value)
        parameter_values_CD_prob.append(str(parameters).replace("\n", ""))
    
    # Gets the callback function for the respective optimizer.
    optimizer.callback = locals().get(f"callback_fun_{optimizer.__class__.__name__}")
    # Sets the optimizer to minimize the expectation value F_P for a given initial point x0.
    result = optimizer.minimize(fun = F_P_estimator, x0 = init_params)
    # Converts final ansatz state circuit into a statevector.
    CD_prob_state = Statevector.from_instruction(candidate_circuit.assign_parameters(result.x))

    return result, CD_prob_state

In [ ]:

# Defines the Schwinger model Hamiltonian for a given J, w, m, theta.
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.
# Outputs the Hamiltonian, the ground state as a statevector, and the ground energy.

def Schwinger_Hamiltonian(num_qubits, J, w, m, theta):

    # Constructs the H_ZZ term.
    Paulis_ZZ = []
    for n in range(1, num_qubits):
        for k in range(1,n):
            Paulis_ZZ.append(("ZZ", [k-1, n-1], J*(num_qubits-n)/2))
    
    H_ZZ = SparsePauliOp.from_sparse_list(Paulis_ZZ, num_qubits)

    # Constructs the H_+- term.    
    Paulis_pm = []
    for n in range(1,num_qubits):
        W_n = w-(-1)**n*m*np.sin(theta)/2
        Paulis_pm.append(("XX", [n-1, n], W_n/2))
        Paulis_pm.append(("YY", [n-1, n], W_n/2))
    
    H_pm = SparsePauliOp.from_sparse_list(Paulis_pm, num_qubits)

    # Constructs the H_Z term.
    Paulis_Z = []
    for n in range(1, num_qubits):
        C_n = np.floor(num_qubits/2) - np.floor(n/2)
        Paulis_Z.append(("Z", [n-1], (-1)**n*m*np.cos(theta)/2-J*C_n/2))
    Paulis_Z.append(("Z", [num_qubits-1], (-1)**num_qubits*m*np.cos(theta)/2))
    
    H_Z = SparsePauliOp.from_sparse_list(Paulis_Z, num_qubits)

    # Adds the terms together to get the full Hamiltonian.
    H_prob_Schwinger = H_ZZ + H_pm + H_Z
    # Removes terms from the Hamiltonian with very small coefficients which are meant to be zero as this can lead to errors in the optimizer.
    H_prob_Schwinger = H_prob_Schwinger.simplify(atol=1e-10)

    Schwinger_ground_state = QuantumCircuit(num_qubits)

    # Constructing the ground state of the Schwinger Hamiltonian by numerically finding the ground state and then using .initialize to
    # initialize a quantum circuit to that state.    
    H_prob_Schwinger_matrix = np.array(H_prob_Schwinger)
    eigenvalues, eigenvectors = np.linalg.eigh(H_prob_Schwinger_matrix)
    Schwinger_ground_state.initialize(eigenvectors[:,0], Schwinger_ground_state.qubits)
    Schwinger_ground_state = Statevector.from_instruction(Schwinger_ground_state)
    # Finds ground energy of the Schwinger Hamiltonian.
    Schwinger_ground_energy = eigenvalues[0]
    
    return H_prob_Schwinger, Schwinger_ground_state, Schwinger_ground_energy

In [ ]:

# Constructs and returns an XY mixer Hamiltonian and its ground state to be used as the initial state in the ansatz state of the algorithms.
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.

def XY_mixer(num_qubits):

    # Constructing the XY mixer Hamiltonian.
    Paulis_mix = []
    for n in range(num_qubits-1):
        Paulis_mix.append(("XX", [n,n+1], 1/2))
        Paulis_mix.append(("YY", [n,n+1], 1/2))
    
    H_mix = SparsePauliOp.from_sparse_list(Paulis_mix, num_qubits)

    # Constructing the ground state of the XY mixer Hamiltonian by numerically finding the ground state and then using .initialize to
    # initialize a quantum circuit to that state.
    initial_state = QuantumCircuit(num_qubits)
    H_mix_matrix = np.array(H_mix)
    eigenvalues, eigenvectors = np.linalg.eigh(H_mix_matrix)
    initial_state.initialize(eigenvectors[:,0], initial_state.qubits)

    return H_mix, initial_state

In [ ]:

# Constructs and returns an X mixer Hamiltonian and its ground state to be used as the initial state in the ansatz state of the algorithms.
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.

def X_mixer(num_qubits):

    # Constructing the X mixer Hamiltonian
    Paulis_mix = []
    for i in range(num_qubits):
        Paulis_mix.append(("X", [i], 1))
    
    H_mix = SparsePauliOp.from_sparse_list(Paulis_mix, num_qubits)
    
    # Creates the ground state of the X mixer Hamiltonian
    initial_state = QuantumCircuit(num_qubits)
    initial_state.h(range(num_qubits))
    initial_state.z(range(num_qubits))
    
    return H_mix, initial_state

In [ ]:

# Constructs and returns an A_DC operator from the operator pool A_XY generated by calculating A_λ for the Schwinger model Hamiltonian with an XY mixer 
# with ℓ=2 and keeping up to three-body terms.
# The operator pool is A_XY = ['XY', 'YX', 'XYZ', 'XZY', 'YXZ', 'YZX', 'ZXY', 'ZYX'].
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.
# The input 'term' determines which operator from the pool is used. E.g. to use the term XY from A_XY, input 'XY' into the function.
# The chosen operator term is summed over all the qubits while preserving the ordering of the operator term.

def A_DC_XY(num_qubits, term):

    # Gives an error message if a operator term entered is not in A_XY.
    if term!='XY' and term!='YX' and term!='ZXY' and term!='ZYX' and term!='XZY' and term!='XYZ' and term!='YZX' and term!='YXZ':
        print(term, 'is not a valid CD term for this Hamiltonian.')
        return

    Paulis_CD = []
    # Constructs A_DC for a two-bodied operator.
    if term=='XY' or term=='YX':
        for n in range(1, num_qubits+1):
            for k in range(1,n):
                Paulis_CD.append((term, [k-1, n-1], 1))

    # Constructs A_DC for a three-bodied operator.
    else:
        for a in range(1, num_qubits+1):
            for b in range(1,a):
                for c in range(1,b):
                    Paulis_CD.append((term, [c-1, b-1, a-1], 1))
       
    A_DC = SparsePauliOp.from_sparse_list(Paulis_CD, num_qubits)
    
    return A_DC

In [ ]:

# Constructs and returns an A_DC operator from the operator pool A_X generated by calculating A_λ for the Schwinger model Hamiltonian with an X mixer 
# with ℓ=2 and keeping up to two-body terms.
# The operator pool is A_X = ['Y', 'XY', 'YX', 'YZ', 'ZY'].
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.
# The input 'term' determine which operator from the pool is used. E.g. to use the term XY from A_X, input 'XY' into the function.
# The chosen operator term is summed over all the qubits while preserving the ordering of the operator term.

def A_DC_X(num_qubits, term):

    # Gives an error message if a operator term entered is not in A_X.
    if term!='Y' and term!='YZ' and term!='ZY' and term!='XY' and term!='YX':
        print(term, 'is not a valid CD term for this Hamiltonian.')
        return

    Paulis_CD = []
    # Constructs A_DC for a one-bodied operator.
    if term=='Y':
        for i in range(num_qubits):
            Paulis_CD.append(("Y", [i], 1))

    # Constructs A_DC for a two-bodied operator.
    else:
        for n in range(1, num_qubits+1):
            for k in range(1,n):
                Paulis_CD.append((term, [k-1, n-1], 1))

    A_DC = SparsePauliOp.from_sparse_list(Paulis_CD, num_qubits)
    
    return A_DC

In [ ]:

# Sets the variables for running the algorithms.

# Sets the number of layers to run the algorithms with.
layers = np.arange(1,6)
# Sets the number of qubits.
num_qubits = 4
# Sets mixer to use which also sets the operator pool for the A_DC operator. Can be an X or XY mixer
H_mix_term = 'XY'
# Sets the bumber of times to run the algorithms.
runs = 20

# Model parameters for Schwinger model.
a = 5/6
a_string = '5/6' # Saved as a string so that the exact fraction appears in plot title and text files.
g = 1
m = 0.35
theta = 0
J = g**2*a/2
w = 1/(2*a)

# Sets the optimizer and the maxiter setting for the optimizer. Possible optimizers that can be used are GradientDescent, COBYLA, and SPSA.
maxiter = 300
optimizer = opt.GradientDescent(maxiter = maxiter)

# String of the type of plot to be used in the plot titles and text files.
plot_type = 'CD algorithms'
# String of the name of the code to be recorded in text files.
code_name = 'Schwinger_CD_algorithms.ipynb'

In [ ]:

# Runs QAOA, DC-QAOA, CD-inspired, CD-mixer, and CD-prob for the Schwinger model.

# Constructing the Schwinger model Hamiltonian and its ground state and ground energy.
H_prob_Schwinger, Schwinger_ground_state, Schwinger_ground_energy = Schwinger_Hamiltonian(num_qubits, J, w, m, theta)

# Constructs the mixer Hamiltonian and the operator pool for the A_DC operator.
if H_mix_term == 'XY':
    
    H_mix, initial_state = XY_mixer(num_qubits)
    A_DC_term_list = ['XY', 'YX', 'XYZ', 'XZY', 'YXZ', 'YZX', 'ZXY', 'ZYX']
    
elif H_mix_term == 'X':
    
    H_mix, initial_state = X_mixer(num_qubits)
    A_DC_term_list = ['Y', 'XY', 'YX', 'YZ', 'ZY']
    
# Setting H_mix_term to something other than 'X' or 'XY' will produce an error.
else:
    print('Not a valid mixer term. This code can only run an X or XY mixer, set H_mix_term to \'XY\' or \'X\'.')
    sys.exit()

# Stores optimizer name and settings for plot title and text files.
optimizer_name = optimizer.__class__.__name__ 
optimizer_settings = rf'{optimizer}, {optimizer.settings}'

# String of the optimizer and maxiter for the text files.
plot_opt_variables = rf"optimizer={optimizer_name}, maxiter={maxiter}"
# String of the model parameters for the plots and text files.
plot_parameters = rf"$a={a_string}, g={g}, J={J:.2}, w={w:.2f}, m={m:.2f}, \theta={theta}$"
# Clean variables have \'s and $'s (which are needed for latex writing in plots) removed for the text files.
plot_parameters_clean = plot_parameters.replace("\\", "").replace("$", "")

# Stores the full optimization (energy) values, parameter values, approximation ratios, and results from running the algorithms.
opt_values_full_QAOA = []
parameter_values_full_QAOA = []
approx_ratios_QAOA = []
results_full_QAOA = []

opt_values_full_DC_QAOA = []
parameter_values_full_DC_QAOA = []
approx_ratios_DC_QAOA = []
results_full_DC_QAOA = []

opt_values_full_CD_inspired = []
parameter_values_full_CD_inspired = []
approx_ratios_CD_inspired = []
results_full_CD_inspired = []

opt_values_full_CD_mixer = []
parameter_values_full_CD_mixer = []
approx_ratios_CD_mixer = []
results_full_CD_mixer = []

opt_values_full_CD_prob = []
parameter_values_full_CD_prob = []
approx_ratios_CD_prob = []
results_full_CD_prob = []

# Stores the best optimization (energy) values, parameter values, approximation ratios, and results from running the algorithms.
opt_values_QAOA_best = []
parameter_values_QAOA_best = []
approx_ratios_QAOA_best = []
results_QAOA_best = []

opt_values_DC_QAOA_best = []
parameter_values_DC_QAOA_best = []
approx_ratios_DC_QAOA_best = []
results_DC_QAOA_best = []

opt_values_CD_inspired_best = []
parameter_values_CD_inspired_best = []
approx_ratios_CD_inspired_best = []
results_CD_inspired_best = []

opt_values_CD_mixer_best = []
parameter_values_CD_mixer_best = []
approx_ratios_CD_mixer_best = []
results_CD_mixer_best = []

opt_values_CD_prob_best = []
parameter_values_CD_prob_best = []
approx_ratios_CD_prob_best = []
results_CD_prob_best = []

# Stores the initial variational parameters.
init_params_full_QAOA = []
init_params_full_DC_QAOA = []
init_params_full_CD_inspired = []
init_params_full_CD_mixer = []
init_params_full_CD_prob = []

# Sets the parameters that are passed into the next layer.
set_params_QAOA = []
set_params_DC_QAOA = []
set_params_CD_inspired = []
set_params_CD_mixer = []
set_params_CD_prob = []

# A_DC_term_set contains the fixed A_DC terms built up layer by layer.
# A_DC_set contains the fixed A_DC Hamiltonians built up layer by layer.
# A_DC_full contains the A_DC terms that the CD algorithms are ran with to be used in text files.
A_DC_term_set_DC_QAOA = []
A_DC_set_DC_QAOA = []
A_DC_full_DC_QAOA = []

A_DC_term_set_CD_inspired = []
A_DC_set_CD_inspired = []
A_DC_full_CD_inspired = []

A_DC_term_set_CD_mixer = []
A_DC_set_CD_mixer = []
A_DC_full_CD_mixer = []

A_DC_term_set_CD_prob = []
A_DC_set_CD_prob = []
A_DC_full_CD_prob = []

# Stores the A_DC_term_list index of the fixed CD term for when calculating the average optimization (energy) values.
DC_QAOA_fixed_A_DC_index = []
CD_inspired_fixed_A_DC_index = []
CD_mixer_fixed_A_DC_index = []
CD_prob_fixed_A_DC_index = []

# Stores the minimum value of the expectation value found during an algorithm run.
minimum_values_full_QAOA = []
minimum_values_full_DC_QAOA = []
minimum_values_full_CD_inspired = []
minimum_values_full_CD_mixer = []
minimum_values_full_CD_prob = []

# Stores the average minimum value of the expectation value found for the individual A_DC terms and layers.
minimum_term_avg_full_DC_QAOA = []
minimum_term_avg_full_CD_inspired = []
minimum_term_avg_full_CD_mixer = []
minimum_term_avg_full_CD_prob = []

# Stores the set parameters that are passed into the next layer.
set_params_full_QAOA = []
set_params_full_DC_QAOA = []
set_params_full_CD_inspired = []
set_params_full_CD_mixer = []
set_params_full_CD_prob = []

# Stores the state fidelities from the end of all the runs of the algorithms.
state_fidelity_QAOA = []
state_fidelity_DC_QAOA = []
state_fidelity_CD_inspired = []
state_fidelity_CD_mixer = []
state_fidelity_CD_prob = []

# Stores the average state fidelities for the individual A_DC terms and layers.
state_fidelity_avg_full_DC_QAOA = []
state_fidelity_avg_full_CD_inspired = []
state_fidelity_avg_full_CD_mixer = []
state_fidelity_avg_full_CD_prob = []

# Stores the best state fidelities found.
state_fidelity_full_QAOA_best = []
state_fidelity_full_DC_QAOA_best = []
state_fidelity_full_CD_inspired_best = []
state_fidelity_full_CD_mixer_best = []
state_fidelity_full_CD_prob_best = []

# Stores the average state fidelities of the fixed A_DC terms for each layer (and average state fidelity for each layer for QAOA).
state_fidelity_fixed_QAOA = []
state_fidelity_fixed_DC_QAOA = []
state_fidelity_fixed_CD_inspired = []
state_fidelity_fixed_CD_mixer = []
state_fidelity_fixed_CD_prob = []

# Store the A_DC and run index of the run with the best approximation ratio for each layer.
QAOA_min_indices = []
DC_QAOA_min_indices = []
CD_inspired_min_indices = []
CD_mixer_min_indices = []
CD_prob_min_indices = []

# Store the A_DC and run index of the run with the best state fidelity for each layer.
QAOA_sf_indices = []
DC_QAOA_sf_indices = []
CD_inspired_sf_indices = []
CD_mixer_sf_indices = []
CD_prob_sf_indices = []

# Loops over layers.
for i in layers:

    # Stores the optimization (energy) values, parameter values and approximation ratios for all the runs for a layer.
    opt_values_runs_QAOA = []
    parameter_values_runs_QAOA = []
    results_runs_QAOA = []
    minimum_value_QAOA = []
    init_params_runs_QAOA = []
    state_fidelity_runs_QAOA = []

    # Sets up the list to have an empty list for each A_DC term in the operator pool.
    opt_values_runs_DC_QAOA = [[] for _ in range(len(A_DC_term_list))]
    parameter_values_runs_DC_QAOA = [[] for _ in range(len(A_DC_term_list))]
    results_runs_DC_QAOA = [[] for _ in range(len(A_DC_term_list))]
    minimum_value_DC_QAOA = [[] for _ in range(len(A_DC_term_list))]
    init_params_runs_DC_QAOA = [[] for _ in range(len(A_DC_term_list))]
    A_DC_runs_DC_QAOA = [[] for _ in range(len(A_DC_term_list))]
    state_fidelity_runs_DC_QAOA = [[] for _ in range(len(A_DC_term_list))]

    opt_values_runs_CD_inspired = [[] for _ in range(len(A_DC_term_list))]
    parameter_values_runs_CD_inspired = [[] for _ in range(len(A_DC_term_list))]
    results_runs_CD_inspired = [[] for _ in range(len(A_DC_term_list))]
    minimum_value_CD_inspired = [[] for _ in range(len(A_DC_term_list))]
    init_params_runs_CD_inspired = [[] for _ in range(len(A_DC_term_list))]
    A_DC_runs_CD_inspired = [[] for _ in range(len(A_DC_term_list))]
    state_fidelity_runs_CD_inspired = [[] for _ in range(len(A_DC_term_list))]

    opt_values_runs_CD_mixer = [[] for _ in range(len(A_DC_term_list))]
    parameter_values_runs_CD_mixer = [[] for _ in range(len(A_DC_term_list))]
    results_runs_CD_mixer = [[] for _ in range(len(A_DC_term_list))]
    minimum_value_CD_mixer = [[] for _ in range(len(A_DC_term_list))]
    init_params_runs_CD_mixer = [[] for _ in range(len(A_DC_term_list))]
    A_DC_runs_CD_mixer = [[] for _ in range(len(A_DC_term_list))]
    state_fidelity_runs_CD_mixer = [[] for _ in range(len(A_DC_term_list))]

    opt_values_runs_CD_prob = [[] for _ in range(len(A_DC_term_list))]
    parameter_values_runs_CD_prob = [[] for _ in range(len(A_DC_term_list))]
    results_runs_CD_prob = [[] for _ in range(len(A_DC_term_list))]
    minimum_value_CD_prob = [[] for _ in range(len(A_DC_term_list))]
    init_params_runs_CD_prob = [[] for _ in range(len(A_DC_term_list))]
    A_DC_runs_CD_prob = [[] for _ in range(len(A_DC_term_list))]
    state_fidelity_runs_CD_prob = [[] for _ in range(len(A_DC_term_list))]

    # Loops over the number of the parameters.
    for x in range(runs):

        print(rf'Starting layer {i}, run {x+1}.')

        # Sets the initial variantional parameters for the algorithms. 
        # The parameters are set according to the parameter fixing strategy in arXiv:2108.05288.
        init_params_QAOA = [set_params_QAOA]
        init_params_DC_QAOA = [set_params_DC_QAOA]
        init_params_CD_inspired = [set_params_CD_inspired]
        init_params_CD_mixer = [set_params_CD_mixer]
        init_params_CD_prob = [set_params_CD_prob]

        # The random parameters are set to random values between 0 and 2pi. The same random parameters are passed to all the algorithms.
        alpha = 2 * np.pi * np.random.random()
        beta = 2 * np.pi * np.random.random()
        gamma = 2 * np.pi * np.random.random()

        init_params_QAOA = np.insert(init_params_QAOA, i-1, beta)
        init_params_QAOA = np.append(init_params_QAOA, gamma)

        init_params_DC_QAOA = np.insert(init_params_DC_QAOA, i-1, alpha)
        init_params_DC_QAOA = np.insert(init_params_DC_QAOA, 2*i-1, beta)
        init_params_DC_QAOA = np.append(init_params_DC_QAOA, gamma)

        init_params_CD_inspired = np.append(init_params_CD_inspired, alpha)

        init_params_CD_mixer = np.insert(init_params_CD_mixer, i-1, alpha)
        init_params_CD_mixer = np.append(init_params_CD_mixer, beta)

        init_params_CD_prob = np.insert(init_params_CD_prob, i-1, alpha)
        init_params_CD_prob = np.append(init_params_CD_prob, gamma)
        
        # Runs QAOA and stores the initial parameter values, optimization values, parameter values, optimizer results, 
        # the final ansatz state, the minimum value, and the state fidelity of the run.
        init_params_runs_QAOA.append(init_params_QAOA)
        opt_values_QAOA = []
        parameter_values_QAOA = []
        results_QAOA, QAOA_state = QAOA(H_prob_Schwinger, H_mix, initial_state, i, init_params_QAOA, num_qubits, optimizer)
        # Gets the minimum of the expectation value F_P that was found.
        opt_values_runs_QAOA.append(opt_values_QAOA)
        parameter_values_runs_QAOA.append(parameter_values_QAOA)
        minimum_value_QAOA.append(results_QAOA.fun)
        results_runs_QAOA.append(results_QAOA)
        state_fidelity_runs_QAOA.append(state_fidelity(QAOA_state, Schwinger_ground_state))

        # Indexes the lists for the different A_DC terms.
        count = 0
        # Loops over the A_DC terms in the operator pool.
        for A_DC_term in A_DC_term_list:

            print(rf'Starting {A_DC_term} runs.')        

            # Sets up the A_DC terms for DC-QAOA. The term with the best average approximation ratio gets fixed as the A_DC term for that layer.
            A_DC_terms_DC_QAOA = A_DC_term_set_DC_QAOA.copy()
            A_DC_terms_DC_QAOA.append(A_DC_term)
            A_DC_runs_DC_QAOA[count].append(A_DC_terms_DC_QAOA)           
            A_DC_DC_QAOA = A_DC_set_DC_QAOA.copy()
            A_DC_DC_QAOA.append(A_DC_XY(num_qubits, A_DC_term))

            # Runs DC-QAOA and stores the initial parameter values, optimization values, parameter values, optimizer results, 
            # the final ansatz state, the minimum value, and the state fidelity of the run.
            init_params_runs_DC_QAOA[count].append(init_params_DC_QAOA)
            opt_values_DC_QAOA = []
            parameter_values_DC_QAOA = []
            results_DC_QAOA, DC_QAOA_state = DC_QAOA(H_prob_Schwinger, H_mix, A_DC_DC_QAOA, initial_state, i, init_params_DC_QAOA, num_qubits, optimizer)
            opt_values_runs_DC_QAOA[count].append(opt_values_DC_QAOA)
            parameter_values_runs_DC_QAOA[count].append(parameter_values_DC_QAOA)
            minimum_value_DC_QAOA[count].append(results_DC_QAOA.fun)
            results_runs_DC_QAOA[count].append(results_DC_QAOA)
            state_fidelity_runs_DC_QAOA[count].append(state_fidelity(DC_QAOA_state, Schwinger_ground_state))

            # Sets up the A_DC terms for CD-inspired. The term with the best average approximation ratio gets fixed as the A_DC term for that layer.
            A_DC_terms_CD_inspired = A_DC_term_set_CD_inspired.copy()
            A_DC_terms_CD_inspired.append(A_DC_term)
            A_DC_runs_CD_inspired[count].append(A_DC_terms_CD_inspired)           
            A_DC_CD_inspired = A_DC_set_CD_inspired.copy()
            A_DC_CD_inspired.append(A_DC_XY(num_qubits, A_DC_term))

            # Runs CD-inspired and stores the initial parameter values, optimization values, parameter values, optimizer results, 
            # the final ansatz state, the minimum value, and the state fidelity of the run.
            init_params_runs_CD_inspired[count].append(init_params_CD_inspired)
            opt_values_CD_inspired = []
            parameter_values_CD_inspired = []
            results_CD_inspired, CD_inspired_state = CD_inspired(H_prob_Schwinger, A_DC_CD_inspired, initial_state, i, init_params_CD_inspired, num_qubits, optimizer)
            opt_values_runs_CD_inspired[count].append(opt_values_CD_inspired)
            parameter_values_runs_CD_inspired[count].append(parameter_values_CD_inspired)
            minimum_value_CD_inspired[count].append(results_CD_inspired.fun)
            results_runs_CD_inspired[count].append(results_CD_inspired)
            state_fidelity_runs_CD_inspired[count].append(state_fidelity(CD_inspired_state, Schwinger_ground_state))

            # Sets up the A_DC terms for CD-mixer. The term with the best average approximation ratio gets fixed as the A_DC term for that layer.
            A_DC_terms_CD_mixer = A_DC_term_set_CD_mixer.copy()
            A_DC_terms_CD_mixer.append(A_DC_term)
            A_DC_runs_CD_mixer[count].append(A_DC_terms_CD_mixer)           
            A_DC_CD_mixer = A_DC_set_CD_mixer.copy()
            A_DC_CD_mixer.append(A_DC_XY(num_qubits, A_DC_term))

            # Runs CD-mixer and stores the initial parameter values, optimization values, parameter values, optimizer results, 
            # the final ansatz state, the minimum value, and the state fidelity of the run.
            init_params_runs_CD_mixer[count].append(init_params_CD_mixer)
            opt_values_CD_mixer = []
            parameter_values_CD_mixer = []
            results_CD_mixer, CD_mixer_state = CD_mixer(H_prob_Schwinger, H_mix, A_DC_CD_mixer, initial_state, i, init_params_CD_mixer, num_qubits, optimizer)
            opt_values_runs_CD_mixer[count].append(opt_values_CD_mixer)
            parameter_values_runs_CD_mixer[count].append(parameter_values_CD_mixer)
            minimum_value_CD_mixer[count].append(results_CD_mixer.fun)
            results_runs_CD_mixer[count].append(results_CD_mixer)
            state_fidelity_runs_CD_mixer[count].append(state_fidelity(CD_mixer_state, Schwinger_ground_state))

            # Sets up the A_DC terms for CD-prob. The term with the best average approximation ratio gets fixed as the A_DC term for that layer.
            A_DC_terms_CD_prob = A_DC_term_set_CD_prob.copy()
            A_DC_terms_CD_prob.append(A_DC_term)
            A_DC_runs_CD_prob[count].append(A_DC_terms_CD_prob)           
            A_DC_CD_prob = A_DC_set_CD_prob.copy()
            A_DC_CD_prob.append(A_DC_XY(num_qubits, A_DC_term))

            # Runs CD-prob and stores the initial parameter values, optimization values, parameter values, optimizer results, 
            # the final ansatz state, the minimum value, and the state fidelity of the run.
            init_params_runs_CD_prob[count].append(init_params_CD_prob)
            opt_values_CD_prob = []
            parameter_values_CD_prob = []
            results_CD_prob, CD_prob_state = CD_prob(H_prob_Schwinger, A_DC_CD_prob, initial_state, i, init_params_CD_prob, num_qubits, optimizer)
            opt_values_runs_CD_prob[count].append(opt_values_CD_prob)
            parameter_values_runs_CD_prob[count].append(parameter_values_CD_prob)
            minimum_value_CD_prob[count].append(results_CD_prob.fun)
            results_runs_CD_prob[count].append(results_CD_prob)
            state_fidelity_runs_CD_prob[count].append(state_fidelity(CD_prob_state, Schwinger_ground_state))

            count += 1

    # Finds the lowest minimum value index, highest state fidelity index, and sets the parameters for the next layer for QAOA.
    QAOA_min_index = minimum_value_QAOA.index(min(minimum_value_QAOA))
    QAOA_min_indices.append(QAOA_min_index)
    QAOA_sf_index = state_fidelity_runs_QAOA.index(max(state_fidelity_runs_QAOA))
    QAOA_sf_indices.append(QAOA_sf_index)
    set_params_QAOA = results_runs_QAOA[QAOA_min_index].x
    set_params_full_QAOA.append(set_params_QAOA)

    # Takes the average of the minimum values and state fidelities for each A_DC term for DC-QAOA.
    minimum_term_avg_DC_QAOA = []
    state_fidelity_avg_DC_QAOA = []
    for i in range(len(A_DC_term_list)):
        minimum_term_avg_DC_QAOA.append(np.average(minimum_value_DC_QAOA[i]))
        state_fidelity_avg_DC_QAOA.append(np.average(state_fidelity_runs_DC_QAOA[i]))
    minimum_term_avg_full_DC_QAOA.append(minimum_term_avg_DC_QAOA)
    state_fidelity_avg_full_DC_QAOA.append(state_fidelity_avg_DC_QAOA)

    # Finds the index for the A_DC term with the lowest average minimum value for DC-QAOA.
    DC_QAOA_term_min_index_1 = minimum_term_avg_DC_QAOA.index(min(minimum_term_avg_DC_QAOA))
    DC_QAOA_term_min_index_2 = minimum_value_DC_QAOA[DC_QAOA_term_min_index_1].index(min(minimum_value_DC_QAOA[DC_QAOA_term_min_index_1]))
    DC_QAOA_fixed_A_DC_index.append(DC_QAOA_term_min_index_1)

    # Finds the lowest minimum value and its index for DC-QAOA.
    DC_QAOA_min_index_1, DC_QAOA_min_index_2, minimum_value_DC_QAOA_best = min(
        ((DC_QAOA_min_index_1, DC_QAOA_min_index_2, minimum_value_DC_QAOA_best)
         for DC_QAOA_min_index_1, sublist in enumerate(minimum_value_DC_QAOA)
         for DC_QAOA_min_index_2, minimum_value_DC_QAOA_best in enumerate(sublist)),
        key=lambda t: t[2])
    DC_QAOA_min_indices.append([DC_QAOA_min_index_1, DC_QAOA_min_index_2])

    # Finds the highest state fidelity and its index for DC-QAOA.
    DC_QAOA_sf_index_1, DC_QAOA_sf_index_2, state_fidelity_DC_QAOA_best = max(
        ((DC_QAOA_sf_index_1, DC_QAOA_sf_index_2, state_fidelity_DC_QAOA_best)
         for DC_QAOA_sf_index_1, sublist in enumerate(state_fidelity_runs_DC_QAOA)
         for DC_QAOA_sf_index_2, state_fidelity_DC_QAOA_best in enumerate(sublist)),
        key=lambda t: t[2])
    DC_QAOA_sf_indices.append([DC_QAOA_sf_index_1, DC_QAOA_sf_index_2])

    # Sets the parameters for the next layer and fixes the A_DC term for this layer for DC-QAOA.
    set_params_DC_QAOA = results_runs_DC_QAOA[DC_QAOA_term_min_index_1][DC_QAOA_term_min_index_2].x
    set_params_full_DC_QAOA.append(set_params_DC_QAOA)
    A_DC_term_set_DC_QAOA.append(A_DC_runs_DC_QAOA[DC_QAOA_term_min_index_1][DC_QAOA_term_min_index_2][-1])
    A_DC_set_DC_QAOA.append(A_DC_XY(num_qubits, A_DC_term_set_DC_QAOA[-1]))
    A_DC_full_DC_QAOA.append(A_DC_runs_DC_QAOA)

    # Takes the average of the minimum values and state fidelities for each A_DC term for CD-inspired.
    minimum_term_avg_CD_inspired = []
    state_fidelity_avg_CD_inspired = []
    for i in range(len(A_DC_term_list)):
        minimum_term_avg_CD_inspired.append(np.average(minimum_value_CD_inspired[i]))
        state_fidelity_avg_CD_inspired.append(np.average(state_fidelity_runs_CD_inspired[i]))
    minimum_term_avg_full_CD_inspired.append(minimum_term_avg_CD_inspired)
    state_fidelity_avg_full_CD_inspired.append(state_fidelity_avg_CD_inspired)

    # Finds the index for the A_DC term with the lowest average minimum value for CD-inspired.
    CD_inspired_term_min_index_1 = minimum_term_avg_CD_inspired.index(min(minimum_term_avg_CD_inspired))
    CD_inspired_term_min_index_2 = minimum_value_CD_inspired[CD_inspired_term_min_index_1].index(min(minimum_value_CD_inspired[CD_inspired_term_min_index_1]))
    CD_inspired_fixed_A_DC_index.append(CD_inspired_term_min_index_1)

    # Finds the lowest minimum value and its index for CD-inspired.
    CD_inspired_min_index_1, CD_inspired_min_index_2, minimum_value_CD_inspired_best = min(
        ((CD_inspired_min_index_1, CD_inspired_min_index_2, minimum_value_CD_inspired_best)
         for CD_inspired_min_index_1, sublist in enumerate(minimum_value_CD_inspired)
         for CD_inspired_min_index_2, minimum_value_CD_inspired_best in enumerate(sublist)),
        key=lambda t: t[2])
    CD_inspired_min_indices.append([CD_inspired_min_index_1, CD_inspired_min_index_2])

    # Finds the highest state fidelity and its index for CD-inspired.
    CD_inspired_sf_index_1, CD_inspired_sf_index_2, state_fidelity_CD_inspired_best = max(
        ((CD_inspired_sf_index_1, CD_inspired_sf_index_2, state_fidelity_CD_inspired_best)
         for CD_inspired_sf_index_1, sublist in enumerate(state_fidelity_runs_CD_inspired)
         for CD_inspired_sf_index_2, state_fidelity_CD_inspired_best in enumerate(sublist)),
        key=lambda t: t[2])
    CD_inspired_sf_indices.append([CD_inspired_sf_index_1, CD_inspired_sf_index_2])

    # Sets the parameters for the next layer and fixes the A_DC term for this layer for CD-inspired.
    set_params_CD_inspired = results_runs_CD_inspired[CD_inspired_term_min_index_1][CD_inspired_term_min_index_2].x
    set_params_full_CD_inspired.append(set_params_CD_inspired)
    A_DC_term_set_CD_inspired.append(A_DC_runs_CD_inspired[CD_inspired_term_min_index_1][CD_inspired_term_min_index_2][-1])
    A_DC_set_CD_inspired.append(A_DC_XY(num_qubits, A_DC_term_set_CD_inspired[-1]))
    A_DC_full_CD_inspired.append(A_DC_runs_CD_inspired)

    # Takes the average of the minimum values and state fidelities for each A_DC term for CD-mixer.
    minimum_term_avg_CD_mixer = []
    state_fidelity_avg_CD_mixer = []
    for i in range(len(A_DC_term_list)):
        minimum_term_avg_CD_mixer.append(np.average(minimum_value_CD_mixer[i]))
        state_fidelity_avg_CD_mixer.append(np.average(state_fidelity_runs_CD_mixer[i]))
    minimum_term_avg_full_CD_mixer.append(minimum_term_avg_CD_mixer)
    state_fidelity_avg_full_CD_mixer.append(state_fidelity_avg_CD_mixer)

    # Finds the index for the A_DC term with the lowest average minimum value for CD-mixer.
    CD_mixer_term_min_index_1 = minimum_term_avg_CD_mixer.index(min(minimum_term_avg_CD_mixer))
    CD_mixer_term_min_index_2 = minimum_value_CD_mixer[CD_mixer_term_min_index_1].index(min(minimum_value_CD_mixer[CD_mixer_term_min_index_1]))
    CD_mixer_fixed_A_DC_index.append(CD_mixer_term_min_index_1)

    # Finds the lowest minimum value and its index for CD-mixer.
    CD_mixer_min_index_1, CD_mixer_min_index_2, minimum_value_CD_mixer_best = min(
        ((CD_mixer_min_index_1, CD_mixer_min_index_2, minimum_value_CD_mixer_best)
         for CD_mixer_min_index_1, sublist in enumerate(minimum_value_CD_mixer)
         for CD_mixer_min_index_2, minimum_value_CD_mixer_best in enumerate(sublist)),
        key=lambda t: t[2])
    CD_mixer_min_indices.append([CD_mixer_min_index_1, CD_mixer_min_index_2])

    # Finds the highest state fidelity and its index for CD-mixer.
    CD_mixer_sf_index_1, CD_mixer_sf_index_2, state_fidelity_CD_mixer_best = max(
        ((CD_mixer_sf_index_1, CD_mixer_sf_index_2, state_fidelity_CD_mixer_best)
         for CD_mixer_sf_index_1, sublist in enumerate(state_fidelity_runs_CD_mixer)
         for CD_mixer_sf_index_2, state_fidelity_CD_mixer_best in enumerate(sublist)),
        key=lambda t: t[2])
    CD_mixer_sf_indices.append([CD_mixer_sf_index_1, CD_mixer_sf_index_2])

    # Sets the parameters for the next layer and fixes the A_DC term for this layer for CD-mixer.
    set_params_CD_mixer = results_runs_CD_mixer[CD_mixer_term_min_index_1][CD_mixer_term_min_index_2].x
    set_params_full_CD_mixer.append(set_params_CD_mixer)
    A_DC_term_set_CD_mixer.append(A_DC_runs_CD_mixer[CD_mixer_term_min_index_1][CD_mixer_term_min_index_2][-1])
    A_DC_set_CD_mixer.append(A_DC_XY(num_qubits, A_DC_term_set_CD_mixer[-1]))
    A_DC_full_CD_mixer.append(A_DC_runs_CD_mixer)

    # Takes the average of the minimum values and state fidelities for each A_DC term for CD-prob.
    minimum_term_avg_CD_prob = []
    state_fidelity_avg_CD_prob = []
    for i in range(len(A_DC_term_list)):
        minimum_term_avg_CD_prob.append(np.average(minimum_value_CD_prob[i]))
        state_fidelity_avg_CD_prob.append(np.average(state_fidelity_runs_CD_prob[i]))
    minimum_term_avg_full_CD_prob.append(minimum_term_avg_CD_prob)
    state_fidelity_avg_full_CD_prob.append(state_fidelity_avg_CD_prob)

    # Finds the index for the A_DC term with the lowest average minimum value for CD-prob.
    CD_prob_term_min_index_1 = minimum_term_avg_CD_prob.index(min(minimum_term_avg_CD_prob))
    CD_prob_term_min_index_2 = minimum_value_CD_prob[CD_prob_term_min_index_1].index(min(minimum_value_CD_prob[CD_prob_term_min_index_1]))
    CD_prob_fixed_A_DC_index.append(CD_prob_term_min_index_1)

    # Finds the lowest minimum value and its index for CD-prob.
    CD_prob_min_index_1, CD_prob_min_index_2, minimum_value_CD_prob_best = min(
        ((CD_prob_min_index_1, CD_prob_min_index_2, minimum_value_CD_prob_best)
         for CD_prob_min_index_1, sublist in enumerate(minimum_value_CD_prob)
         for CD_prob_min_index_2, minimum_value_CD_prob_best in enumerate(sublist)),
        key=lambda t: t[2])
    CD_prob_min_indices.append([CD_prob_min_index_1, CD_prob_min_index_2])

    # Finds the highest state fidelity and its index for CD-prob.
    CD_prob_sf_index_1, CD_prob_sf_index_2, state_fidelity_CD_prob_best = max(
        ((CD_prob_sf_index_1, CD_prob_sf_index_2, state_fidelity_CD_prob_best)
         for CD_prob_sf_index_1, sublist in enumerate(state_fidelity_runs_CD_prob)
         for CD_prob_sf_index_2, state_fidelity_CD_prob_best in enumerate(sublist)),
        key=lambda t: t[2])
    CD_prob_sf_indices.append([CD_prob_sf_index_1, CD_prob_sf_index_2])

    # Sets the parameters for the next layer and fixes the A_DC term for this layer for CD-prob.
    set_params_CD_prob = results_runs_CD_prob[CD_prob_term_min_index_1][CD_prob_term_min_index_2].x
    set_params_full_CD_prob.append(set_params_CD_prob)
    A_DC_term_set_CD_prob.append(A_DC_runs_CD_prob[CD_prob_term_min_index_1][CD_prob_term_min_index_2][-1])
    A_DC_set_CD_prob.append(A_DC_XY(num_qubits, A_DC_term_set_CD_prob[-1]))
    A_DC_full_CD_prob.append(A_DC_runs_CD_prob)

    # Appends the results, optimization values, parameter values, minimum values, initial parameters, average approximation ratios,
    # state fidelity, and average state fidelity for the runs from QAOA.
    results_full_QAOA.append(results_runs_QAOA)
    opt_values_full_QAOA.append(opt_values_runs_QAOA)
    parameter_values_full_QAOA.append(parameter_values_runs_QAOA)
    minimum_values_full_QAOA.append(minimum_value_QAOA)
    init_params_full_QAOA.append(init_params_runs_QAOA)
    approx_ratios_QAOA.append(np.average(minimum_value_QAOA)/Schwinger_ground_energy)
    state_fidelity_QAOA.append(state_fidelity_runs_QAOA)
    state_fidelity_fixed_QAOA.append(np.average(state_fidelity_runs_QAOA))

    # Appends the results, optimization values, parameter values, minimum values, initial parameters, state fidelity, 
    # and average approximation ratios and average state fidelity for the fixed A_DC term for the runs from DC-QAOA.
    results_full_DC_QAOA.append(results_runs_DC_QAOA)
    opt_values_full_DC_QAOA.append(opt_values_runs_DC_QAOA)
    parameter_values_full_DC_QAOA.append(parameter_values_runs_DC_QAOA)
    minimum_values_full_DC_QAOA.append(minimum_value_DC_QAOA)
    init_params_full_DC_QAOA.append(init_params_runs_DC_QAOA)
    approx_ratios_DC_QAOA.append(minimum_term_avg_DC_QAOA[DC_QAOA_term_min_index_1]/Schwinger_ground_energy)
    state_fidelity_DC_QAOA.append(state_fidelity_runs_DC_QAOA)
    state_fidelity_fixed_DC_QAOA.append(state_fidelity_avg_DC_QAOA[DC_QAOA_term_min_index_1])

    # Appends the results, optimization values, parameter values, minimum values, initial parameters, state fidelity, 
    # and average approximation ratios and average state fidelity for the fixed A_DC term for the runs from CD-inspired.
    results_full_CD_inspired.append(results_runs_CD_inspired)
    opt_values_full_CD_inspired.append(opt_values_runs_CD_inspired)
    parameter_values_full_CD_inspired.append(parameter_values_runs_CD_inspired)
    minimum_values_full_CD_inspired.append(minimum_value_CD_inspired)
    init_params_full_CD_inspired.append(init_params_runs_CD_inspired)
    approx_ratios_CD_inspired.append(minimum_term_avg_CD_inspired[CD_inspired_term_min_index_1]/Schwinger_ground_energy)
    state_fidelity_CD_inspired.append(state_fidelity_runs_CD_inspired)
    state_fidelity_fixed_CD_inspired.append(state_fidelity_avg_CD_inspired[CD_inspired_term_min_index_1])

    # Appends the results, optimization values, parameter values, minimum values, initial parameters, state fidelity, 
    # and average approximation ratios and average state fidelity for the fixed A_DC term for the runs from CD-mixer.
    results_full_CD_mixer.append(results_runs_CD_mixer)
    opt_values_full_CD_mixer.append(opt_values_runs_CD_mixer)
    parameter_values_full_CD_mixer.append(parameter_values_runs_CD_mixer)
    minimum_values_full_CD_mixer.append(minimum_value_CD_mixer)
    init_params_full_CD_mixer.append(init_params_runs_CD_mixer)
    approx_ratios_CD_mixer.append(minimum_term_avg_CD_mixer[CD_mixer_term_min_index_1]/Schwinger_ground_energy)
    state_fidelity_CD_mixer.append(state_fidelity_runs_CD_mixer)
    state_fidelity_fixed_CD_mixer.append(state_fidelity_avg_CD_mixer[CD_mixer_term_min_index_1])

    # Appends the results, optimization values, parameter values, minimum values, initial parameters, state fidelity, 
    # and average approximation ratios and average state fidelity for the fixed A_DC term for the runs from CD-prob.
    results_full_CD_prob.append(results_runs_CD_prob)
    opt_values_full_CD_prob.append(opt_values_runs_CD_prob)
    parameter_values_full_CD_prob.append(parameter_values_runs_CD_prob)
    minimum_values_full_CD_prob.append(minimum_value_CD_prob)
    init_params_full_CD_prob.append(init_params_runs_CD_prob)
    approx_ratios_CD_prob.append(minimum_term_avg_CD_prob[CD_prob_term_min_index_1]/Schwinger_ground_energy)
    state_fidelity_CD_prob.append(state_fidelity_runs_CD_prob)
    state_fidelity_fixed_CD_prob.append(state_fidelity_avg_CD_prob[CD_prob_term_min_index_1])

    # Appends the highest state fidelity, highest approximation ratio and the optimization values, parameter values, 
    # and results from the that run for QAOA. 
    results_QAOA_best.append(results_runs_QAOA[QAOA_min_index])
    opt_values_QAOA_best.append(opt_values_runs_QAOA[QAOA_min_index])
    parameter_values_QAOA_best.append(parameter_values_runs_QAOA[QAOA_min_index])
    approx_ratios_QAOA_best.append(min(minimum_value_QAOA)/Schwinger_ground_energy)
    state_fidelity_full_QAOA_best.append(max(state_fidelity_runs_QAOA))

    # Appends the highest state fidelity, highest approximation ratio and the optimization values, parameter values, 
    # and results from the that run for DC-QAOA.
    results_DC_QAOA_best.append(results_runs_DC_QAOA[DC_QAOA_min_index_1][DC_QAOA_min_index_2])
    opt_values_DC_QAOA_best.append(opt_values_runs_DC_QAOA[DC_QAOA_min_index_1][DC_QAOA_min_index_2])
    parameter_values_DC_QAOA_best.append(parameter_values_runs_DC_QAOA[DC_QAOA_min_index_1][DC_QAOA_min_index_2])
    approx_ratios_DC_QAOA_best.append(minimum_value_DC_QAOA_best/Schwinger_ground_energy)
    state_fidelity_full_DC_QAOA_best.append(state_fidelity_DC_QAOA_best)

    # Appends the highest state fidelity, highest approximation ratio and the optimization values, parameter values, 
    # and results from the that run for CD-inspired.
    results_CD_inspired_best.append(results_runs_CD_inspired[CD_inspired_min_index_1][CD_inspired_min_index_2])
    opt_values_CD_inspired_best.append(opt_values_runs_CD_inspired[CD_inspired_min_index_1][CD_inspired_min_index_2])
    parameter_values_CD_inspired_best.append(parameter_values_runs_CD_inspired[CD_inspired_min_index_1][CD_inspired_min_index_2])
    approx_ratios_CD_inspired_best.append(minimum_value_CD_inspired_best/Schwinger_ground_energy)
    state_fidelity_full_CD_inspired_best.append(state_fidelity_CD_inspired_best)

    # Appends the highest state fidelity, highest approximation ratio and the optimization values, parameter values, 
    # and results from the that run for CD-mixer.
    results_CD_mixer_best.append(results_runs_CD_mixer[CD_mixer_min_index_1][CD_mixer_min_index_2])
    opt_values_CD_mixer_best.append(opt_values_runs_CD_mixer[CD_mixer_min_index_1][CD_mixer_min_index_2])
    parameter_values_CD_mixer_best.append(parameter_values_runs_CD_mixer[CD_mixer_min_index_1][CD_mixer_min_index_2])
    approx_ratios_CD_mixer_best.append(minimum_value_CD_mixer_best/Schwinger_ground_energy)
    state_fidelity_full_CD_mixer_best.append(state_fidelity_CD_mixer_best)

    # Appends the highest state fidelity, highest approximation ratio and the optimization values, parameter values, 
    # and results from the that run for CD-prob.
    results_CD_prob_best.append(results_runs_CD_prob[CD_prob_min_index_1][CD_prob_min_index_2])
    opt_values_CD_prob_best.append(opt_values_runs_CD_prob[CD_prob_min_index_1][CD_prob_min_index_2])
    parameter_values_CD_prob_best.append(parameter_values_runs_CD_prob[CD_prob_min_index_1][CD_prob_min_index_2])
    approx_ratios_CD_prob_best.append(minimum_value_CD_prob_best/Schwinger_ground_energy)
    state_fidelity_full_CD_prob_best.append(state_fidelity_CD_prob_best)

# Appends the optimization values, paramete values, approximation ratios, optimizer results, initial parameters, state fidelities, minimum values,
# average minimum values for A_DC terms, average state fidelities, average state fidelities for the fixed A_DC terms, the A_DC terms used, 
# the set parameters from each of the algorithms, the fixed A_DC index into a single list.
plots_opt_full = []
plots_parameter_full = []
plots_ar_full = []
results_full = []
init_params_full = []
state_fidelity_full = []
minimum_values_full = []
minimum_term_values_full = []
state_fidelity_avg_full = []
state_fidelity_fixed_full = []
A_DC_full = []
set_params_full = []
fixed_A_DC_index_full = []

plots_opt_full.append(opt_values_full_QAOA)
plots_parameter_full.append(parameter_values_full_QAOA)
plots_ar_full.append(approx_ratios_QAOA)
results_full.append(results_full_QAOA)
init_params_full.append(init_params_full_QAOA)
minimum_values_full.append(minimum_values_full_QAOA)
state_fidelity_full.append(state_fidelity_QAOA)
state_fidelity_fixed_full.append(state_fidelity_fixed_QAOA)
set_params_full.append(set_params_full_QAOA)

plots_opt_full.append(opt_values_full_DC_QAOA)
plots_parameter_full.append(parameter_values_full_DC_QAOA)
plots_ar_full.append(approx_ratios_DC_QAOA)
results_full.append(results_full_DC_QAOA)
init_params_full.append(init_params_full_DC_QAOA)
state_fidelity_full.append(state_fidelity_DC_QAOA)
minimum_values_full.append(minimum_values_full_DC_QAOA)
minimum_term_values_full.append(minimum_term_avg_full_DC_QAOA)
state_fidelity_avg_full.append(state_fidelity_avg_full_DC_QAOA)
state_fidelity_fixed_full.append(state_fidelity_fixed_DC_QAOA)
A_DC_full.append(A_DC_full_DC_QAOA)
set_params_full.append(set_params_full_DC_QAOA)
fixed_A_DC_index_full.append(DC_QAOA_fixed_A_DC_index)

plots_opt_full.append(opt_values_full_CD_inspired)
plots_parameter_full.append(parameter_values_full_CD_inspired)
plots_ar_full.append(approx_ratios_CD_inspired)
results_full.append(results_full_CD_inspired)
init_params_full.append(init_params_full_CD_inspired)
state_fidelity_full.append(state_fidelity_CD_inspired)
minimum_values_full.append(minimum_values_full_CD_inspired)
minimum_term_values_full.append(minimum_term_avg_full_CD_inspired)
state_fidelity_avg_full.append(state_fidelity_avg_full_CD_inspired)
state_fidelity_fixed_full.append(state_fidelity_fixed_CD_inspired)
A_DC_full.append(A_DC_full_CD_inspired)
set_params_full.append(set_params_full_CD_inspired)
fixed_A_DC_index_full.append(CD_inspired_fixed_A_DC_index)

plots_opt_full.append(opt_values_full_CD_mixer)
plots_parameter_full.append(parameter_values_full_CD_mixer)
plots_ar_full.append(approx_ratios_CD_mixer)
results_full.append(results_full_CD_mixer)
init_params_full.append(init_params_full_CD_mixer)
state_fidelity_full.append(state_fidelity_CD_mixer)
minimum_values_full.append(minimum_values_full_CD_mixer)
minimum_term_values_full.append(minimum_term_avg_full_CD_mixer)
state_fidelity_avg_full.append(state_fidelity_avg_full_CD_mixer)
state_fidelity_fixed_full.append(state_fidelity_fixed_CD_mixer)
A_DC_full.append(A_DC_full_CD_mixer)
set_params_full.append(set_params_full_CD_mixer)
fixed_A_DC_index_full.append(CD_mixer_fixed_A_DC_index)

plots_opt_full.append(opt_values_full_CD_prob)
plots_parameter_full.append(parameter_values_full_CD_prob)
plots_ar_full.append(approx_ratios_CD_prob)
results_full.append(results_full_CD_prob)
init_params_full.append(init_params_full_CD_prob)
state_fidelity_full.append(state_fidelity_CD_prob)
minimum_values_full.append(minimum_values_full_CD_prob)
minimum_term_values_full.append(minimum_term_avg_full_CD_prob)
state_fidelity_avg_full.append(state_fidelity_avg_full_CD_prob)
state_fidelity_fixed_full.append(state_fidelity_fixed_CD_prob)
A_DC_full.append(A_DC_full_CD_prob)
set_params_full.append(set_params_full_CD_prob)
fixed_A_DC_index_full.append(CD_prob_fixed_A_DC_index)

# Appends the highest state fidelity, highest state fidelity index, highest approximation ratio and the optimization values, parameter values
# results from that run, and the highest approximation ratio index from each of the algorithms into a single list.
plots_opt_full_best = []
plots_parameter_full_best = []
plots_ar_full_best = []
results_full_best = []
state_fidelity_avg_full_best = []
min_indices_full = []
sf_indices_full = []

plots_opt_full_best.append(opt_values_QAOA_best)
plots_parameter_full_best.append(parameter_values_QAOA_best)
plots_ar_full_best.append(approx_ratios_QAOA_best)
results_full_best.append(results_QAOA_best)
state_fidelity_avg_full_best.append(state_fidelity_full_QAOA_best)
min_indices_full.append(QAOA_min_indices)
sf_indices_full.append(QAOA_sf_indices)

plots_opt_full_best.append(opt_values_DC_QAOA_best)
plots_parameter_full_best.append(parameter_values_DC_QAOA_best)
plots_ar_full_best.append(approx_ratios_DC_QAOA_best)
results_full_best.append(results_DC_QAOA_best)
state_fidelity_avg_full_best.append(state_fidelity_full_DC_QAOA_best)
min_indices_full.append(DC_QAOA_min_indices)
sf_indices_full.append(DC_QAOA_sf_indices)

plots_opt_full_best.append(opt_values_CD_inspired_best)
plots_parameter_full_best.append(parameter_values_CD_inspired_best)
plots_ar_full_best.append(approx_ratios_CD_inspired_best)
results_full_best.append(results_CD_inspired_best)
state_fidelity_avg_full_best.append(state_fidelity_full_CD_inspired_best)
min_indices_full.append(CD_inspired_min_indices)
sf_indices_full.append(CD_inspired_sf_indices)

plots_opt_full_best.append(opt_values_CD_mixer_best)
plots_parameter_full_best.append(parameter_values_CD_mixer_best)
plots_ar_full_best.append(approx_ratios_CD_mixer_best)
results_full_best.append(results_CD_mixer_best)
state_fidelity_avg_full_best.append(state_fidelity_full_CD_mixer_best)
min_indices_full.append(CD_mixer_min_indices)
sf_indices_full.append(CD_mixer_sf_indices)

plots_opt_full_best.append(opt_values_CD_prob_best)
plots_parameter_full_best.append(parameter_values_CD_prob_best)
plots_ar_full_best.append(approx_ratios_CD_prob_best)
results_full_best.append(results_CD_prob_best)
state_fidelity_avg_full_best.append(state_fidelity_full_CD_prob_best)
min_indices_full.append(CD_prob_min_indices)
sf_indices_full.append(CD_prob_sf_indices)

# Stores the subplot titles.
subplot_titles = ['QAOA', 'DC-QAOA', 'CD-inspired', 'CD-mixer', 'CD-prob']

# Add timestamp to plot file name to prevent plots from being saved over.
timestamp_EP_full = datetime.now().strftime("%Y%m%d_%H%M%S")
# Filename of the text file.
filename_subplot_EP_full = rf"Schwinger model {plot_type}, {num_qubits} qubits, {H_mix_term} mixer, full data_{timestamp_EP_full}"

# Creates text file to store data of all the plots.
with open(filename_subplot_EP_full + ".txt", "w") as f:
    
    f.write(rf"Schwinger model {plot_type} full data" "\n" rf"{num_qubits} qubits, {H_mix_term} mixer, {runs} runs, {plot_opt_variables}, {plot_parameters_clean}, minimum energy = {Schwinger_ground_energy}")
    f.write("\n\n")
    
    f.write("=" * 50 + "\n")
    f.write(subplot_titles[0] + "\n")
    f.write("=" * 50 + "\n\n")

    for j in range(layers[-1]):

        f.write("-" * 50 + "\n")
        f.write(f"Layer = {j+1}\n")
        f.write("-" * 50 + "\n\n")

        for k in range(runs):

            f.write("*" * 50 + "\n")
            f.write(f"Run {k+1}" + "\n")
            f.write(f"Initial random parameters = {init_params_full[0][j][k]}\n")
            f.write("*" * 50 + "\n\n")
            
            f.write("Iteration:    Cost:                       Parameters:\n\n")

            iteration = 0
            for cost, parameters in zip(plots_opt_full[0][j][k], plots_parameter_full[0][j][k]):
                f.write(f"{iteration:<14}{cost:<28}{parameters}\n")
                iteration +=1

            f.write("\n")
            f.write("Optimization results:\n\n")
            f.write(f"{results_full[0][j][k]}\n")

            f.write("\n")
            f.write(f"Approximation ratio: {minimum_values_full[0][j][k]/Schwinger_ground_energy}\n")
            f.write(f"State fidelity: {state_fidelity_full[0][j][k]}\n\n")
            
        f.write("." * 50 + "\n")  
        f.write(f'Set parameters: {set_params_full[0][j]}\n')
        f.write("." * 50 + "\n\n")

    f.write("\n")
    f.write("~" * 50 + "\n")
    f.write(f"Average results over {runs} runs\n")
    f.write("~" * 50 + "\n\n")
    f.write(rf"Layer:    Approximation ratio:        State fidelity" "\n\n")                                                                                                                                                                                                                 

    for l in range(layers[-1]):
        f.write(f"{l+1:<10}{plots_ar_full[0][l]:<28}{state_fidelity_fixed_full[0][l]}\n")

    f.write("\n\n")

    f.write("\n")
    f.write("~" * 50 + "\n")
    f.write(f"Best results over {runs} runs\n")
    f.write("~" * 50 + "\n\n")
    f.write(rf"Layer:    Approximation ratio:                 State fidelity:" "\n\n")                                                                                                                                                                                                                 

    for l in range(layers[-1]):
        f.write(f"{l+1:<10}{plots_ar_full_best[0][l]} from run {min_indices_full[0][l]+1:<9}{state_fidelity_avg_full_best[0][l]} from run {sf_indices_full[0][l]+1:<28}\n")

    f.write("\n\n")

    for i in range(1, 5):
        
        f.write("=" * 50 + "\n")
        f.write(subplot_titles[i] + "\n")
        f.write("=" * 50 + "\n\n")
    
        for j in range(layers[-1]):
    
            f.write("-" * 50 + "\n")
            f.write(f"Layer = {j+1}\n")
            f.write("-" * 50 + "\n\n")

            for l in range(len(A_DC_term_list)):
    
                for k in range(runs):
        
                    f.write("*" * 50 + "\n")
                    f.write(f"A_DC terms = {A_DC_full[i-1][j][l][k]}, Run {k+1}" + "\n")
                    f.write(f"Initial random parameters = {init_params_full[i][j][l][k]}\n")
                    f.write("*" * 50 + "\n\n")
                    
                    f.write("Iteration:    Cost:                       Parameters:\n\n")
        
                    iteration = 0
                    for cost, parameters in zip(plots_opt_full[i][j][l][k], plots_parameter_full[i][j][l][k]):
                        f.write(f"{iteration:<14}{cost:<28}{parameters}\n")
                        iteration +=1
        
                    f.write("\n")
                    f.write("Optimization results:\n\n")
                    f.write(f"{results_full[i][j][l][k]}\n\n")

                    f.write("\n")
                    f.write(f"Approximation ratio: {minimum_values_full[i][j][l][k]/Schwinger_ground_energy}\n")
                    f.write(f"State fidelity: {state_fidelity_full[i][j][l][k]}\n\n")

                f.write("#" * 50 + "\n")
                f.write(f'Term average approximation ratio over {runs} runs: {minimum_term_values_full[i-1][j][l]/Schwinger_ground_energy}\n')
                f.write(f'Term average state fidelity over {runs} runs: {state_fidelity_avg_full[i-1][j][l]}\n')
                f.write("#" * 50 + "\n\n")

            f.write("." * 50 + "\n")
            f.write(f'Set parameters: {set_params_full[i][j]}\n')
            f.write("." * 50 + "\n\n")
            
        f.write("\n")
        f.write("~" * 50 + "\n")
        f.write(f"Best term average results over {runs} runs\n")
        f.write("~" * 50 + "\n\n")
        f.write(rf"Layer:    Approximation ratio:        State fidelity" "\n\n")                                                                                                                                                                                                                 
    
        for l in range(layers[-1]):
            f.write(f"{l+1:<10}{plots_ar_full[i][l]:<27}{state_fidelity_fixed_full[i][l]}\n")
    
        f.write("\n\n")

        f.write("\n")
        f.write("+" * 50 + "\n")
        f.write(f"Best results over {runs} runs\n")
        f.write("+" * 50 + "\n\n")
        f.write(rf"Layer:    Approximation ratio:                            State fidelity:" "\n\n")                                                                                                                                                                                                                 
    
        for l in range(layers[-1]):
            f.write(f"{l+1:<10}{plots_ar_full_best[i][l]} from {A_DC_term_list[min_indices_full[i][l][0]]} term, run {min_indices_full[i][l][1]+1:<12}{state_fidelity_avg_full_best[i][l]} from {A_DC_term_list[sf_indices_full[i][l][0]]} term, run {sf_indices_full[i][l][1]+1:<28}\n")
    
        f.write("\n\n")

    f.write("=" * 50 + "\n")
    f.write(f"Final CD term set\n")
    f.write("=" * 50 + "\n\n")

    f.write(f"DC-QAOA: {A_DC_term_set_DC_QAOA}\n")
    f.write(f"CD-inspired: {A_DC_term_set_CD_inspired}\n")
    f.write(f"CD-Mixer: {A_DC_term_set_CD_mixer}\n")
    f.write(f"CD_prob: {A_DC_term_set_CD_prob}\n")

    f.write("\n\n")        


In [ ]:
# Averages the optimization values across the runs and stores the data for average energy plots.
# For the CD algorithms, the optimization values are only averaged for the fixed A_DC terms runs.

plots_opt_avg_full = []

avg_values_layers = []

for a in range(len(layers)):

    avg_values = []
    
    for x in range(maxiter):
        
        run_values = []
        
        for y in range(runs):
            
            try:
                plots_opt_full[0][a][y][x]
                
            except IndexError:
                run_values.append(plots_opt_full[0][a][y][-1])
                
            else:
                run_values.append(plots_opt_full[0][a][y][x])

        avg_values.append(np.average(run_values))

    avg_values_layers.append(avg_values)

plots_opt_avg_full.append(avg_values_layers)

for i in range(1,5):

    avg_values_layers = []
        
    for a in range(np.size(layers)):
    
        avg_values = []
        
        for x in range(maxiter):
    
            run_values = []
            
            for y in range(runs):
                
                try:
                    plots_opt_full[i][a][fixed_A_DC_index_full[i-1][a]][y][x]
                    
                except IndexError:
                    run_values.append(plots_opt_full[i][a][fixed_A_DC_index_full[i-1][a]][y][x])
                    
                else:
                    run_values.append(plots_opt_full[i][a][fixed_A_DC_index_full[i-1][a]][y][x])
    
            avg_values.append(np.average(run_values))
    
        avg_values_layers.append(avg_values)
    
    plots_opt_avg_full.append(avg_values_layers)

In [ ]:
# Sets up the subplots for average and best energy plots.

# Number of plots in figures.
n_plots = 5
# Every row has 5 columns.
ncols = 5
# Calculates the number of rows.
nrows = int(np.ceil(n_plots / ncols))
# Sets the position of certain objects on the figure.
supy_y = 0.39
legend_y = 0.92
tight_top = 0.9

# Sets the font size of the x and y tick labels.
plt.rc('xtick', labelsize=20) 
plt.rc('ytick', labelsize=20)

In [ ]:
# Plots the average energy for each of the algorithms in a subplot. Saves the data in a text file.

# Setting up the figure.
fig_EP, axes_EP = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows+1.2), sharex=True, sharey=True)
fig_EP.suptitle(rf"Schwinger model {plot_type} average energy plots - {num_qubits} qubits, {H_mix_term} mixer, {plot_parameters}", fontsize=25)
plt.subplots_adjust(wspace=0.05, hspace=0.05)
axes_EP = axes_EP.flatten()

# Plots the data.
for i in range(5):
    
    for j in range(layers[-1]):
        axes_EP[i].plot(plots_opt_avg_full[i][j], '.', label=rf"p={j+1}")
        axes_EP[i].set_title(rf"{subplot_titles[i]}", fontsize=20)

    # Plots the ground energy.
    axes_EP[i].axhline(y=Schwinger_ground_energy, linestyle="--", color='k', linewidth=1, label="Exact ground energy")

# Hides any unused subplots
for j in range(n_plots, len(axes_EP)):
    axes_EP[j].set_visible(False)

# Turns on axis for the lower most plot in each column (this is due to hiding the unused plots).
for col in range(ncols):

    col_axes_EP = axes_EP[col::ncols]
    visible_axes_EP = [ax for ax in col_axes_EP if ax.get_visible()]
    
    if visible_axes_EP:
        bottom_ax = visible_axes_EP[-1]
        bottom_ax.tick_params(labelbottom=True)

# Axes labels.
fig_EP.supxlabel("Iteration", fontsize=20, x=0.525, y=0.025)
fig_EP.supylabel(rf"Average energy", fontsize=20, x=0.0075, y=supy_y)

# Creates the legend.
handles, labels = axes_EP[0].get_legend_handles_labels()
fig_EP.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, legend_y), ncol=layers[-1]+1, fontsize=20)

plt.tight_layout(rect=[0, 0, 1, tight_top])
plt.show()

# Add timestamp to plot file name to prevent plots from being saved over.
timestamp_EP = datetime.now().strftime("%Y%m%d_%H%M%S")
filename_subplot_EP = rf"Schwinger model {plot_type}, {num_qubits} qubits, {H_mix_term} mixer, average energy plots_{timestamp_EP}"
# Saves the figure.
fig_EP.savefig(filename_subplot_EP + ".png", bbox_inches="tight") # Comment out if you don't want the figure saved

# Creates text file to store data of the plots.
with open(filename_subplot_EP + ".txt", "w") as f:
    
    f.write(rf"Schwinger model {plot_type} average energy plots" "\n" rf"{num_qubits} qubits, {H_mix_term} mixer, {runs} runs, {plot_opt_variables}, {plot_parameters_clean}, minimum energy = {Schwinger_ground_energy}")
    f.write("\n\n")
    f.write(rf'Optimizer default settings (apart from maxiter) - {optimizer_settings}')
    f.write("\n\n")
    
    for i in range(n_plots):
        
        f.write("=" * 50 + "\n")
        f.write(subplot_titles[i] + "\n")
        f.write("=" * 50 + "\n\n")

        for j in range(layers[-1]):

            f.write(f"Layer = {j+1}:\n\n")
            f.write(f"Iteration:    Average energy over {runs} runs:\n\n")

            iteration = 0
            for cost in plots_opt_avg_full[i][j]:
                f.write(f"{iteration:<14}{cost}\n")
                iteration +=1
            f.write("\n")
            
        f.write("\n")


In [ ]:
# Plots the energy from the runs with the highest approximation ratio for each of the algorithms in a subplot. Saves the data in a text file.

# Setting up the figure.
fig_EP_best, axes_EP_best = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows+1.2), sharex=True, sharey=True)
fig_EP_best.suptitle(rf"Schwinger model {plot_type} best energy plots - {num_qubits} qubits, {H_mix_term} mixer, {plot_parameters}", fontsize=25)
plt.subplots_adjust(wspace=0.05, hspace=0.05)
axes_EP_best = axes_EP_best.flatten()

# Plots the data.
for i in range(5):
    
    for j in range(layers[-1]):
        axes_EP_best[i].plot(plots_opt_full_best[i][j], '.', label=rf"p={j+1}")
        axes_EP_best[i].set_title(rf"{subplot_titles[i]}", fontsize=20)

    # Plots the ground energy.
    axes_EP_best[i].axhline(y=Schwinger_ground_energy, linestyle="--", color='k', linewidth=1, label="Exact ground energy") 

# Hides any unused subplots
for j in range(n_plots, len(axes_EP_best)):
    axes_EP_best[j].set_visible(False)

# Turns on axis for the lower most plot in each column (this is due to hiding the unused plots).
for col in range(ncols):

    col_axes_EP_best = axes_EP_best[col::ncols]
    visible_axes_EP_best = [ax for ax in col_axes_EP_best if ax.get_visible()]
    
    if visible_axes_EP_best:
        bottom_ax = visible_axes_EP_best[-1]
        bottom_ax.tick_params(labelbottom=True)

# Axes labels.
fig_EP_best.supxlabel("Iteration", fontsize=20, x=0.525, y=0.025)
fig_EP_best.supylabel(rf"Best energy", fontsize=20, x=0.0075, y=supy_y)

# Creates the legend.
handles, labels = axes_EP_best[0].get_legend_handles_labels()
fig_EP_best.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, legend_y), ncol=layers[-1]+1, fontsize=20)

plt.tight_layout(rect=[0, 0, 1, tight_top])
plt.show()

# Add timestamp to plot file name to prevent plots from being saved over.
timestamp_EP_best = datetime.now().strftime("%Y%m%d_%H%M%S")
filename_subplot_EP_best = rf"Schwinger model {plot_type}, {num_qubits} qubits, {H_mix_term} mixer, best energy plots_{timestamp_EP_best}"
# Saves the figure.
fig_EP_best.savefig(filename_subplot_EP_best + ".png", bbox_inches="tight") # Comment out if you don't want the figure saved

# Creates text file to store data of the plots.
with open(filename_subplot_EP_best + ".txt", "w") as f:
    
    f.write(rf"Schwinger model {plot_type} best energy plots" "\n" rf"{num_qubits} qubits, {H_mix_term} mixer, {runs} runs, {plot_opt_variables}, {plot_parameters_clean}, minimum energy = {Schwinger_ground_energy}")
    f.write("\n\n")
    f.write(rf'Optimizer default settings (apart from maxiter) - {optimizer_settings}')
    f.write("\n\n")
        
    f.write("=" * 50 + "\n")
    f.write(subplot_titles[0] + "\n")
    f.write("=" * 50 + "\n\n")

    for j in range(layers[-1]):

        f.write("-" * 50 + "\n")
        f.write(f'Layer {j+1}, from run {min_indices_full[0][j]+1}\n')
        f.write("-" * 50 + "\n\n")
    
        f.write("Iteration:    Cost:                       Parameters:\n\n")
    
        iteration = 0
        for cost, parameters in zip(plots_opt_full_best[0][j], plots_parameter_full_best[0][j]):
            f.write(f"{iteration:<14}{cost:<28}{parameters}\n")
            iteration +=1
    
        f.write("\n")
        f.write("Optimization results:\n\n")
        f.write(f"{results_full_best[i][j]}\n\n")
    
    for i in range(1, n_plots):
        
        f.write("=" * 50 + "\n")
        f.write(subplot_titles[i] + "\n")
        f.write("=" * 50 + "\n\n")

        for j in range(layers[-1]):

            f.write("-" * 50 + "\n")
            f.write(f'Layer {j+1}, from {A_DC_term_list[min_indices_full[i][j][0]]} term, run {min_indices_full[i][j][1]+1}\n')
            f.write("-" * 50 + "\n\n")

            f.write("Iteration:    Cost:                       Parameters:\n\n")
        
            iteration = 0
            for cost, parameters in zip(plots_opt_full_best[i][j], plots_parameter_full_best[i][j]):
                f.write(f"{iteration:<14}{cost:<28}{parameters}\n")
                iteration +=1

            f.write("\n")
            f.write("Optimization results:\n\n")
            f.write(f"{results_full_best[i][j]}\n\n")
            
        f.write("\n")


In [ ]:
# Plots the average approximation ratios for each of the algorithms. Saves the data in a text file.
# The average approximation ratios for the CD algorithm are from the A_DC terms with the highest average approximation ratio.

# Set the different dots style for the different algorithms.
dot_styles = ['o', '*', "^", "s", "D"]
# Setting up the figure.
plt.figure(figsize=(10, 5))
# Plots approximation ratio = 1 line.
plt.axhline(y=1, linestyle="dashdot", color='k', linewidth=2, label="Approximation ratio=1")
plt.xlabel("Number of layers", fontsize=15)
plt.ylabel(rf"Average approximation ratio", fontsize=15)
plt.title(rf"Schwinger model {plot_type} average approximation ratios - {num_qubits} qubits, {H_mix_term} mixer, {plot_parameters}", y=1.2, fontsize=15)
plt.xticks(layers, fontsize=15)
plt.yticks(fontsize=15)
ax = plt.gca()
# Sets the gaps between the y tick labels to 0.1.
ax.yaxis.set_major_locator(MultipleLocator(0.1))

# Plots the data.
for i in range(5):
    plt.plot(layers, plots_ar_full[i], dot_styles[i], linestyle="--", label=rf'{subplot_titles[i]}')

# Creates the legend.
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.225), ncol=3, frameon=True, fontsize=15)

# Add timestamp to plot file name to prevent plots from being saved over.
timestamp_AR = datetime.now().strftime("%Y%m%d_%H%M%S")
filename_subplot_AR = rf"Schwinger model {plot_type}, {num_qubits} qubits, {H_mix_term} mixer, average approximation ratios_{timestamp_EP}"
# Saves the figure.
plt.savefig(filename_subplot_AR + ".png", bbox_inches="tight")

# Creates text file to store data of the plots.
with open(filename_subplot_AR + ".txt", "w") as f:
    
    f.write(rf"Schwinger model {plot_type} average approximation ratios" "\n" rf"{num_qubits} qubits, {H_mix_term} mixer, {runs} runs, {plot_opt_variables}, {plot_parameters_clean}, minimum energy = {Schwinger_ground_energy}")
    f.write("\n\n")
    f.write(rf'Optimizer default settings (apart from maxiter) - {optimizer_settings}')
    f.write("\n\n")
    
    for i in range(n_plots):
        
        f.write("=" * 50 + "\n")
        f.write(subplot_titles[i] + "\n")
        f.write("=" * 50 + "\n\n")

        f.write(rf"Layer:    Approximation ratio:" "\n\n")                                                                                                                                                                                                                 

        for j in range(layers[-1]):
            
            f.write(f"{j+1:<10}{plots_ar_full[i][j]}\n")

        f.write("\n")

In [ ]:
# Plots the highest approximation ratios for each of the algorithms. Saves the data in a text file.

# Set the different dots style for the different algorithms.
dot_styles = ['o', '*', "^", "s", "D"]
# Setting up the figure.
plt.figure(figsize=(10, 5))
# Plots approximation ratio = 1 line.
plt.axhline(y=1, linestyle="dashdot", color='k', linewidth=2, label="Approximation ratio=1")
plt.xlabel("Number of layers", fontsize=15)
plt.ylabel(rf"Best approximation ratio", fontsize=15)
plt.title(rf"Schwinger model {plot_type} best approximation ratios - {num_qubits} qubits, {H_mix_term} mixer, {plot_parameters}", y=1.2, fontsize=15)
plt.xticks(layers, fontsize=15)
plt.yticks(fontsize=15)
ax = plt.gca()
# Sets the gaps between the y tick labels to 0.1.
ax.yaxis.set_major_locator(MultipleLocator(0.1))

# Plots the data.
for i in range(n_plots):
    plt.plot(layers, plots_ar_full_best[i], dot_styles[i], linestyle="--", label=rf'{subplot_titles[i]}')

# Creates the legend.
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.225), ncol=3, frameon=True, fontsize=15)

# Add timestamp to plot file name to prevent plots from being saved over.
timestamp_AR_best = datetime.now().strftime("%Y%m%d_%H%M%S")
filename_subplot_AR_best = rf"Schwinger model {plot_type}, {num_qubits} qubits, {H_mix_term} mixer, best approximation ratios_{timestamp_AR_best}"
# Saves the figure.
plt.savefig(filename_subplot_AR_best + ".png", bbox_inches="tight")

# Creates text file to store data of all the plots.
with open(filename_subplot_AR_best + ".txt", "w") as f:
    
    f.write(rf"Schwinger model {plot_type} best approximation ratios" "\n" rf"{num_qubits} qubits, {H_mix_term} mixer, {runs} runs, {plot_opt_variables}, {plot_parameters_clean}, minimum energy = {Schwinger_ground_energy}")
    f.write("\n\n")
    f.write(rf'Optimizer default settings (apart from maxiter) - {optimizer_settings}')
    f.write("\n\n")

    f.write("=" * 50 + "\n")
    f.write(subplot_titles[0] + "\n")
    f.write("=" * 50 + "\n\n")

    f.write(rf"Layer:    Approximation ratio:" "\n\n")                                                                                                                                                                                                                 

    for j in range(layers[-1]):
        
        f.write("-" * 50 + "\n")
        f.write(f'From run {min_indices_full[0][j]+1:<28}\n')
        f.write("-" * 50 + "\n\n")
        
        f.write(f"{j+1:<10}{plots_ar_full_best[0][j]}\n\n")

    for i in range(1, n_plots):
        
        f.write("=" * 50 + "\n")
        f.write(subplot_titles[i] + "\n")
        f.write("=" * 50 + "\n\n")

        f.write(rf"Layer:    Approximation ratio:" "\n\n")                                                                                                                                                                                                                 

        for j in range(layers[-1]):
            
            f.write("-" * 50 + "\n")
            f.write(f'From {A_DC_term_list[min_indices_full[i][j][0]]} term, run {min_indices_full[i][j][1]+1:<28}\n')
            f.write("-" * 50 + "\n\n")
            
            f.write(f"{j+1:<10}{plots_ar_full_best[i][j]}\n\n")

        f.write("\n")

In [ ]:
# Plots the average state fidelities for each of the algorithms. Saves the data in a text file.
# The average state fidelities for the CD algorithm are from the A_DC terms with the highest average approximation ratio.

# Set the different dots style for the different algorithms.
dot_styles = ['o', '*', "^", "s", "D"]
# Setting up the figure.
plt.figure(figsize=(10, 5))
# Plots state fidelity = 1 line.
plt.axhline(y=1, linestyle="dashdot", color='k', linewidth=2, label="State fidelity=1")
plt.xlabel("Number of layers", fontsize=15)
plt.ylabel(rf"Average state fidelity", fontsize=15)
plt.title(rf"Schwinger model {plot_type} average state fidelity - {num_qubits} qubits, {H_mix_term} mixer, {plot_parameters}", y=1.2 , fontsize=15)
plt.xticks(layers, fontsize=15)
plt.yticks(fontsize=15)
ax = plt.gca()
# Sets the gaps between the y tick labels to 0.1.
ax.yaxis.set_major_locator(MultipleLocator(0.1))

# Plots the data.
for i in range(5):
    plt.plot(layers, state_fidelity_fixed_full[i], dot_styles[i], linestyle="--", label=rf'{subplot_titles[i]}')

# Creates the legend.
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.225), ncol=3, frameon=True, fontsize=15)

# Add timestamp to plot file name to prevent plots from being saved over.
timestamp_sf = datetime.now().strftime("%Y%m%d_%H%M%S")
filename_subplot_sf = rf"Schwinger model {plot_type}, {num_qubits} qubits, {H_mix_term} mixer, average state fidelity_{timestamp_sf}"
# Saves the figure.
plt.savefig(filename_subplot_sf + ".png", bbox_inches="tight")

# Creates text file to store data of all the plots.
with open(filename_subplot_sf + ".txt", "w") as f:
    
    f.write(rf"Schwinger model {plot_type} average state fidelity" "\n" rf"{num_qubits} qubits, {H_mix_term} mixer, {runs} runs, {plot_opt_variables}, {plot_parameters_clean}, minimum energy = {Schwinger_ground_energy}")
    f.write("\n\n")
    f.write(rf'Optimizer default settings (apart from maxiter) - {optimizer_settings}')
    f.write("\n\n")
    
    for i in range(n_plots):
        
        f.write("=" * 50 + "\n")
        f.write(subplot_titles[i] + "\n")
        f.write("=" * 50 + "\n\n")

        f.write(rf"Layer:    Approximation ratio:" "\n\n")                                                                                                                                                                                                                 

        for j in range(layers[-1]):
            
            f.write(f"{j+1:<10}{state_fidelity_fixed_full[i][j]}\n")

        f.write("\n")

In [ ]:
# Plots the best state fidelity for each of the algorithms. Saves the data in a text file.

# Set the different dots style for the different algorithms.
dot_styles = ['o', '*', "^", "s", "D"]
# Setting up the figure.
plt.figure(figsize=(10, 5))
# Plots the state fidelity = 1 line.
plt.axhline(y=1, linestyle="dashdot", color='k', linewidth=2, label="State fidelity=1")
plt.xlabel("Number of layers", fontsize=15)
plt.ylabel(rf"Best state fidelity", fontsize=15)
plt.title(rf"Schwinger model {plot_type} best state fidelity - {num_qubits} qubits, {H_mix_term} mixer, {plot_parameters}", y=1.2 , fontsize=15)
plt.xticks(layers, fontsize=15)
plt.yticks(fontsize=15)
ax = plt.gca()
# Sets the gaps between the y tick labels to 0.1.
ax.yaxis.set_major_locator(MultipleLocator(0.1))

# Plots the data.
for i in range(n_plots):
    plt.plot(layers, state_fidelity_avg_full_best[i], dot_styles[i], linestyle="--", label=rf'{subplot_titles[i]}')

# Creates the legend.
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.225), ncol=3, frameon=True, fontsize=15)

# Add timestamp to plot file name to prevent plots from being saved over.
timestamp_sf_best = datetime.now().strftime("%Y%m%d_%H%M%S")
filename_subplot_sf_best = rf"Schwinger model {plot_type}, {num_qubits} qubits, {H_mix_term} mixer, best state fidelity_{timestamp_sf_best}"
# Saves the figure.
plt.savefig(filename_subplot_sf_best + ".png", bbox_inches="tight")

# Creates text file to store data of all the plots..
with open(filename_subplot_sf_best + ".txt", "w") as f:
    
    f.write(rf"Schwinger model {plot_type} best state fidelity" "\n" rf"{num_qubits} qubits, {H_mix_term} mixer, {runs} runs, {plot_opt_variables}, {plot_parameters_clean}, minimum energy = {Schwinger_ground_energy}")
    f.write("\n\n")
    f.write(rf'Optimizer default settings (apart from maxiter) - {optimizer_settings}')
    f.write("\n\n")

    f.write("=" * 50 + "\n")
    f.write(subplot_titles[0] + "\n")
    f.write("=" * 50 + "\n\n")

    f.write(rf"Layer:    Approximation ratio:" "\n\n")                                                                                                                                                                                                                 

    for j in range(layers[-1]):
        
        f.write("-" * 50 + "\n")
        f.write(f'From run {sf_indices_full[0][j]+1:<28}\n')
        f.write("-" * 50 + "\n\n")
        
        f.write(f"{j+1:<10}{state_fidelity_avg_full_best[0][j]}\n\n")
    
    for i in range(1, n_plots):
        
        f.write("=" * 50 + "\n")
        f.write(subplot_titles[i] + "\n")
        f.write("=" * 50 + "\n\n")

        f.write(rf"Layer:    Approximation ratio:" "\n\n")                                                                                                                                                                                                                 

        for j in range(layers[-1]):
            
            f.write("-" * 50 + "\n")
            f.write(f'From {A_DC_term_list[sf_indices_full[i][j][0]]} term, run {sf_indices_full[i][j][1]+1:<28}\n')
            f.write("-" * 50 + "\n\n")
            
            f.write(f"{j+1:<10}{state_fidelity_avg_full_best[i][j]}\n\n")

        f.write("\n")